# Serving — Shipping the Fine-Tuned Model with vLLM

**Starts from:** the three adapters produced by `01`, `02` and `03`
**Serves:** `vllm serve` — an OpenAI-compatible HTTP endpoint, PagedAttention, continuous batching
**Hardware:** Colab **L4 or A100** · **Runtime:** ~55 minutes on an L4

---

## What this notebook does

Notebooks 01-03 end the moment `save_pretrained()` returns. That is where most fine-tuning
tutorials stop, and it is roughly two thirds of the way to anything anyone can use. What they
produce is a **LoRA adapter**: 45 MB of low-rank matrices that are meaningless without the exact
base weights they were trained against, in the exact order they were trained in. No serving engine
accepts one as a model.

This notebook covers the rest:

```
three adapters  ──►  one merged checkpoint  ──►  vllm serve  ──►  measured
   45 MB x3          2.5 GB, self-contained      HTTP :8000      and verified
```

1. **Export.** Merge all three adapters into the base weights and write a real checkpoint, then
   read what that checkpoint tells a server — including the two fields that will stop it booting.
2. **Serve.** Start `vllm serve`, wait on `/health`, and talk to it with an ordinary OpenAI client.
3. **Verify.** Re-score the same 200 held-out articles through the server. A fast model that
   answers differently is not the model you trained.
4. **Measure.** TTFT, inter-token latency, p95, and the throughput/latency frontier against
   concurrency — with `model.generate()` measured the same way, on the same GPU, in the same
   session, for contrast.

## The two rules this notebook is built around

**Correctness gates speed.** Section 11 comes before every benchmark for a reason. It is also the
section most likely to surprise you: greedy decoding is *not* bit-reproducible across engines, so
the agreement rate between vLLM and `generate()` will be high and will not be 100%.

**A benchmark that does not fix the output length is measuring output length.** If one system emits
40 tokens and the other emits 120, the tokens/sec comparison between them means nothing. Every
timed cell below pins the generated length on both sides.

## What to expect

vLLM will win, and the interesting part is *why* and *by how much at what concurrency*. At batch 1
both systems are memory-bandwidth-bound and the gap is modest. The gap opens up under load, and
sections 7 and 14 measure the two mechanisms responsible — padding waste and head-of-line blocking
— rather than asserting them.

**Set your runtime to an L4 or A100 first:** Runtime → Change runtime type → L4 GPU.

## 1. Runtime and dependencies

`vllm` is new here and it is a heavier dependency than anything else in this repo. Two things it
does that the other notebooks' pins do not:

- **It repins `torch`.** Notebooks 01-04 all say "torch is deliberately left alone, Colab ships a
  CUDA-matched build". vLLM does not get that courtesy, because it ships precompiled CUDA kernels
  linked against one specific torch ABI. Installing it replaces Colab's torch, takes several
  minutes, and will ask for a runtime restart. Accept the restart, then run the cell again.
- **It is picky about `transformers`.** vLLM 0.19.1's specifier excludes transformers 5.0 through
  5.5.0 but allows everything after. This repo's `transformers==5.15.0` pin satisfies it, so
  nothing gets downgraded. That is luck rather than design, and worth checking whenever either pin
  moves.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no GPU found")

In [ ]:
# Pinned to match requirements-colab.txt. Note what is NOT here: torch. Every other notebook in
# this repo leaves Colab's build alone; vllm brings its own and there is no way around it.
%pip install -q "vllm==0.19.1" "transformers==5.15.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.14.0"
# Same torchao removal as notebooks 01-04, for the same reason: peft's optional torchao
# integration RAISES rather than degrading when it finds a version below its 0.16.0 minimum, and
# the check fires inside get_peft_model(). vllm 0.19.1 does not depend on torchao, so removing it
# is safe here too — checked against the package metadata, not assumed.
%pip uninstall -q -y torchao
print("\nRestart the runtime when Colab asks, then run this cell again and continue.")

### This notebook does not run on a T4

Notebooks 01-03 all branch on `SUPPORTS_BF16` and fall back to fp16 on a Turing T4. This one
refuses instead, and the reason is not that it would be slow: **on a T4 you would not be
benchmarking vLLM.**

vLLM's fast attention kernels — FlashAttention, FlashInfer — require compute capability 8.0 or
later. A T4 is Turing, sm_75. vLLM still runs there, by falling back to `TRITON_ATTN` or
`FLEX_ATTENTION`, which are the backends that work everywhere precisely because they make no
architecture-specific assumptions. Numbers collected on that path describe the fallback, not the
engine, and publishing them as "vLLM throughput" would be the systems equivalent of the adapter
that never loaded.

The free-tier promise in this repo's README covers notebooks 01-03. It does not cover this one.

In [ ]:
import asyncio, gc, json, math, random, re, statistics, subprocess, time, urllib.error, urllib.request
from pathlib import Path
from collections import Counter

import torch
import transformers
import peft
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > L4 GPU."

# Notebooks 01-03 fall back to fp16 here. This one refuses — see the cell above.
CAPABILITY = torch.cuda.get_device_capability()
assert CAPABILITY[0] >= 8, (
    f"This notebook needs an Ampere-or-later GPU (sm_80+); found "
    f"sm_{CAPABILITY[0]}{CAPABILITY[1]} ({torch.cuda.get_device_name(0)}). "
    "vLLM's FlashAttention and FlashInfer backends require sm_80, and on a Turing T4 it falls "
    "back to TRITON_ATTN — so every number below would describe the fallback kernels rather than "
    "the engine.\n"
    "Runtime > Change runtime type > L4 GPU (or A100)."
)

DTYPE = torch.bfloat16
DTYPE_BYTES = 2
TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"   # transformers v5 renamed torch_dtype

MODEL_ID = "unsloth/Llama-3.2-1B"
DEVICE_NAME = torch.cuda.get_device_name(0)
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9

import vllm

print(f"vllm {vllm.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")
print(f"torch {torch.__version__}  <- vllm chose this one, not Colab")
print(f"GPU {DEVICE_NAME}  sm_{CAPABILITY[0]}{CAPABILITY[1]}  {TOTAL_VRAM:.1f} GB")

## 2. Find all three adapters

Same probe as notebook 03, extended to stage 3: look for `adapter_config.json` rather than for the
directory, because a half-finished Drive sync leaves an empty folder that looks identical from the
outside.

In [ ]:
# Colab wipes /content on disconnect, so notebooks 01-03 copied their adapters to Drive.
if not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass


def find_adapter(name):
    """Prefer the Drive copy, fall back to this session's local outputs."""
    for candidate in (Path(f"/content/drive/MyDrive/finetuning-demo/{name}"),
                      Path(f"/content/outputs/{name}")):
        if (candidate / "adapter_config.json").exists():
            return candidate
    return None


STAGE1_DIR = find_adapter("stage1-domain-lora")
STAGE2_DIR = find_adapter("stage2-instruct-lora")
STAGE3_DIR = find_adapter("stage3-dpo-lora")

for label, path in [("stage 1 (domain)", STAGE1_DIR), ("stage 2 (instruct)", STAGE2_DIR),
                    ("stage 3 (dpo)", STAGE3_DIR)]:
    print(f"{label:20s} {path if path else 'NOT FOUND'}")

assert STAGE1_DIR and STAGE2_DIR and STAGE3_DIR, (
    "All three adapters are required. Run notebooks 01, 02 and 03 first — this notebook ships the "
    "model they produced, and there is nothing to ship without them."
)

## 3. Export a merged checkpoint

An adapter is not a deployable artifact. `stage3-dpo-lora` is 45 MB of low-rank matrices that mean
nothing without `unsloth/Llama-3.2-1B`, plus stage 1 merged into it, plus stage 2 merged on top of
that — in that order. Hand it to a serving engine and there is nothing for the engine to load.

So: three merges, each verified against a probe weight, exactly the pattern notebook 03 used for
two. What comes out is a plain `LlamaForCausalLM` with all three stages folded into the weights.

**Two checkpoints get written, not one.** The full merge is the thing we serve. But section 16
serves stage 3 as a *hot-swappable* adapter instead of a merged one, and an adapter is only valid
against the weights it was trained against — which for stage 3 means base + stage 1 + stage 2, not
the pristine base. Pointing `--enable-lora` at `unsloth/Llama-3.2-1B` would apply stage 3 to
weights it has never seen. That is the same class of error as loading an adapter with
`AutoModelForCausalLM.from_pretrained()`, and just as quiet. So we snapshot the two-stage model on
the way past.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

EXPORT_DIR = Path("/content/outputs/serving-merged-bf16")      # all three stages, what we serve
LORA_BASE_DIR = Path("/content/outputs/serving-base-s12-bf16") # stages 1+2, the base for section 15

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
assert tokenizer.pad_token_id is not None, "No pad token — you would need to alias EOS here"
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD and EOS must stay distinct"
tokenizer.padding_side = "left"   # section 6 generates in batches; see notebook 03 section 3

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE}).to("cuda")
model.config.pad_token_id = tokenizer.pad_token_id

PROBE_WEIGHT = "model.layers.8.mlp.down_proj.weight"


def probe():
    """Snapshot a weight all three adapters target, so each merge can be proven rather than trusted."""
    return dict(model.named_parameters())[PROBE_WEIGHT].detach().clone()


w = {"base": probe()}
for label, adapter_dir in [("+domain", STAGE1_DIR), ("+instruct", STAGE2_DIR), ("+dpo", STAGE3_DIR)]:
    model = PeftModel.from_pretrained(model, adapter_dir)
    model = model.merge_and_unload()
    w[label] = probe()
    if label == "+instruct":
        model.save_pretrained(LORA_BASE_DIR, safe_serialization=True)
        tokenizer.save_pretrained(LORA_BASE_DIR)

STAGES = ["base", "+domain", "+instruct", "+dpo"]
print(f"probe tensor {PROBE_WEIGHT}")
for a, b in zip(STAGES, STAGES[1:]):
    assert not torch.equal(w[a], w[b]), (
        f"The {b} merge did not change any weights. The adapter was not applied — this is exactly "
        "the failure mode that AutoModelForCausalLM.from_pretrained(adapter_dir) produces, and it "
        "would make every number below describe the wrong model."
    )
    print(f"  {a:10s} -> {b:10s} max |delta| {(w[b].float() - w[a].float()).abs().max():.3e}")

del w
gc.collect(); torch.cuda.empty_cache()

In [ ]:
model.save_pretrained(EXPORT_DIR, safe_serialization=True)
tokenizer.save_pretrained(EXPORT_DIR)

adapter_mb = sum(f.stat().st_size for f in STAGE3_DIR.rglob("*") if f.is_file()) / 1e6
export_mb = sum(f.stat().st_size for f in EXPORT_DIR.rglob("*") if f.is_file()) / 1e6

print(f"{EXPORT_DIR}  ({export_mb:.0f} MB)")
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"  {f.name:34s} {f.stat().st_size / 1e6:8.1f} MB")
print(f"\nstage-3 adapter alone: {adapter_mb:.0f} MB — {adapter_mb / export_mb:.1%} of what a "
      "server needs")

## 4. What the artifact says, and what a server does with it

The checkpoint is written. Before starting anything, read what it is about to tell vLLM, because
two of these fields will stop the server booting and a third will silently ruin every latency
number in the notebook.

Training never had to care about any of them. That is the point: **these are deployment
properties, and they get baked into the artifact by whoever exports it** — which is now you.

In [ ]:
cfg = json.loads((EXPORT_DIR / "config.json").read_text())
gen_cfg_path = EXPORT_DIR / "generation_config.json"
gen_cfg = json.loads(gen_cfg_path.read_text()) if gen_cfg_path.exists() else {}

n_params = sum(p.numel() for p in model.parameters())
WEIGHT_BYTES = n_params * DTYPE_BYTES

# KV cache per token: 2 (K and V) x layers x KV heads x head_dim x bytes.
KV_BYTES_PER_TOKEN = (2 * cfg["num_hidden_layers"] * cfg["num_key_value_heads"]
                      * cfg["head_dim"] * DTYPE_BYTES)

print(f"params              {n_params / 1e9:.3f} B  ({WEIGHT_BYTES / 1e9:.2f} GB in {DTYPE})")
print(f"tie_word_embeddings {cfg.get('tie_word_embeddings')}  <- lm_head shares embed_tokens, so")
print(f"                       it is absent from the safetensors by design")
print(f"torch_dtype         {cfg.get('torch_dtype')}")
print(f"max_position_embeddings {cfg['max_position_embeddings']:,}")
print(f"eos_token_id        config={cfg.get('eos_token_id')}  "
      f"generation_config={gen_cfg.get('eos_token_id')}")
print(f"pad_token_id        config={cfg.get('pad_token_id')}")
print(f"chat template       {'present' if tokenizer.chat_template else 'ABSENT'}")

print(f"\nKV cache: {KV_BYTES_PER_TOKEN / 1024:.0f} KiB per token")
print(f"  a single sequence at the full {cfg['max_position_embeddings']:,}-token context would need "
      f"{cfg['max_position_embeddings'] * KV_BYTES_PER_TOKEN / 1e9:.2f} GB of KV cache")
print(f"  the GPU has {TOTAL_VRAM:.1f} GB in total")

### The three things that just showed up

**1. `max_position_embeddings` is 131,072, and vLLM will believe it.** vLLM defaults `max_model_len`
to whatever the config claims, then checks whether one sequence at that length fits in the KV cache
it was able to reserve. The number printed above says it does not, by a wide margin, so
`vllm serve` would exit during startup with a message about the model's max sequence length
exceeding the available cache. This is not a bug in either tool — Llama 3.2 genuinely supports
128k context, and nothing in fine-tuning ever asked whether we could afford to serve it.

The fix is a serving decision, not a model one: our prompts are at most 896 tokens and we generate
120, so `--max-model-len 1280` is honest and leaves the rest of the memory for concurrency.
Sections 5 and 13 are about what that trade buys.

**2. `torch_dtype` is inherited, not chosen.** It says `bfloat16` because that is what the export
ran in. On an L4 or A100 that is right. Ship this same directory to a Turing box and it is a
performance cliff nobody will attribute to the config file. We pass `--dtype` explicitly to the
server rather than letting it read this field, so the decision is visible in the launch command.

**3. There is no chat template, because this is a base model.** `/v1/chat/completions` needs one to
turn `messages` into a prompt string, so that endpoint will refuse. `/v1/completions` — raw prompt
in, raw text out — is the one to use, and it is the right one anyway: notebooks 02 and 03 trained
on an explicit template with `### Response:`, and letting a chat template wrap it differently at
serving time would produce a prompt the model was never trained on. Section 9 shows the refusal
rather than describing it.

And the one that would not have failed loudly:

In [ ]:
# A wrong or missing EOS means the model never stops on its own, so every request runs to
# max_tokens. Nothing errors. Latency, throughput and cost per token would all be measuring the
# token cap instead of the model, and the numbers would look internally consistent while being
# entirely wrong. Notebook 02 appended EOS to every training target precisely so this works.
assert gen_cfg.get("eos_token_id") is not None or cfg.get("eos_token_id") is not None, (
    "No eos_token_id in the exported checkpoint. The server would generate to max_tokens on "
    "every request and every latency number below would be measuring the cap, not the model."
)
eos_id = gen_cfg.get("eos_token_id", cfg.get("eos_token_id"))
eos_id = eos_id[0] if isinstance(eos_id, list) else eos_id
assert eos_id == tokenizer.eos_token_id, (
    f"Exported eos_token_id ({eos_id}) is not the tokenizer's ({tokenizer.eos_token_id}). The "
    "server would stop on a token the model was never trained to emit."
)
print(f"eos_token_id {eos_id} = {tokenizer.decode([eos_id])!r} — matches the tokenizer")

## 5. Predict the memory before you allocate it

vLLM's headline trick is PagedAttention: the KV cache is managed in fixed-size blocks like virtual
memory pages, so sequences of different lengths pack into it without fragmentation and without
reserving worst-case space per request. What that buys is **concurrency**, and concurrency is
bounded by arithmetic you can do before starting anything.

Two numbers, both checkable later:

**How many sequences fit.** Everything not spent on weights and activations is KV cache. Divide by
the per-token cost and the sequence length, and that is the ceiling on how many requests can be in
flight. Section 10 compares this against what the server reports.

**How fast one sequence can possibly decode.** Generating a token requires reading every weight
once. At batch 1 nothing else is close, so decode is memory-bandwidth-bound and the ceiling is
simply bandwidth divided by model size. Real throughput will be below it; if a measurement ever
comes out *above* it, the measurement is wrong. Section 13 checks that too.

In [ ]:
MAX_MODEL_LEN = 1280        # 896-token prompt cap + 120 generated, with headroom
GPU_MEM_UTIL = 0.85         # fraction of TOTAL VRAM vllm may claim, not fraction of free
MAX_NUM_SEQS = 256          # vllm's own default, stated rather than inherited

# torch exposes no bandwidth attribute, so this is a small lookup. Substitute your own if the
# device is not here — the ratio is what matters, not the exact figure.
GPU_BANDWIDTH_GB_S = {"A100-SXM4-80GB": 2039, "H100": 3350, "A100": 1555, "L40S": 864,
                      "V100": 900, "L4": 300, "T4": 320}
BANDWIDTH = next((v for k, v in sorted(GPU_BANDWIDTH_GB_S.items(), key=lambda kv: -len(kv[0]))
                  if k.lower() in DEVICE_NAME.lower()), None)

# Grouped-query attention is the reason this fits at all: 8 KV heads shared across 32 query heads.
kv_if_mha = (2 * cfg["num_hidden_layers"] * cfg["num_attention_heads"]
             * cfg["head_dim"] * DTYPE_BYTES)

OVERHEAD_GB = 1.0           # activations, CUDA graphs, the engine itself — an estimate, not a read
kv_budget = TOTAL_VRAM * GPU_MEM_UTIL - WEIGHT_BYTES / 1e9 - OVERHEAD_GB
PREDICTED_KV_TOKENS = kv_budget * 1e9 / KV_BYTES_PER_TOKEN
PREDICTED_SEQS = PREDICTED_KV_TOKENS / MAX_MODEL_LEN

print(f"KV per token      {KV_BYTES_PER_TOKEN / 1024:.0f} KiB "
      f"({cfg['num_key_value_heads']} KV heads)")
print(f"  without GQA     {kv_if_mha / 1024:.0f} KiB "
      f"({cfg['num_attention_heads']} heads) — {kv_if_mha / KV_BYTES_PER_TOKEN:.0f}x more")
print(f"\nmemory budget at gpu_memory_utilization={GPU_MEM_UTIL}")
print(f"  total VRAM      {TOTAL_VRAM:6.2f} GB")
print(f"  vllm may claim  {TOTAL_VRAM * GPU_MEM_UTIL:6.2f} GB")
print(f"  weights         {WEIGHT_BYTES / 1e9:6.2f} GB")
print(f"  overhead (est)  {OVERHEAD_GB:6.2f} GB")
print(f"  left for KV     {kv_budget:6.2f} GB")
print(f"\nPREDICTION -> {PREDICTED_KV_TOKENS:,.0f} tokens of KV cache")
print(f"           -> {PREDICTED_SEQS:,.0f} concurrent sequences at max_model_len={MAX_MODEL_LEN}")
print(f"           -> but --max-num-seqs is {MAX_NUM_SEQS}, so "
      f"{'that' if PREDICTED_SEQS > MAX_NUM_SEQS else 'the KV cache'} is the binding limit")

if BANDWIDTH:
    ROOFLINE_TOK_S = BANDWIDTH / (WEIGHT_BYTES / 1e9)
    print(f"\ndecode roofline at batch 1 on {DEVICE_NAME}")
    print(f"  {BANDWIDTH} GB/s / {WEIGHT_BYTES / 1e9:.2f} GB = "
          f"{ROOFLINE_TOK_S:.0f} tokens/sec, absolute ceiling")
else:
    ROOFLINE_TOK_S = None
    print(f"\nNo bandwidth figure for {DEVICE_NAME!r} — the roofline check is skipped.")

## 6. The same 200 held-out records, and the same prompt

Rebuilt exactly as notebooks 02 and 03 build it — same seed, same sort-then-shuffle recipe, same
template. Two different reasons to care:

- Section 11 checks that the served model still scores what notebook 03 measured. That comparison
  is only meaningful against the *same* held-out articles.
- These are also the benchmark workload. Real prompts with a real length distribution beat a
  synthetic `"Hello" * 500`, because prefill cost scales with prompt length and TTFT is mostly
  prefill.

In [ ]:
from datasets import load_dataset

# --- constants mirrored in scripts/prepare_data.py and notebooks 01, 02 and 03 ---
DATASET = "qiaojin/PubMedQA"
SEED = 20260815
EVAL_FRACTION = 0.20      # -> 800 train / 200 eval

TASK = (
    "Answer the research question using only the abstract provided. "
    'Begin your reply with "Answer:" followed by yes, no, or maybe, '
    "then justify it in one or two sentences."
)

labeled = load_dataset(DATASET, "pqa_labeled", split="train")


def join_sections(context) -> str:
    """Render an abstract's sections as `LABEL: text` blocks."""
    labels = context.get("labels") or []
    parts = []
    for i, text in enumerate(context["contexts"]):
        text = " ".join(text.split())
        if not text:
            continue
        label = (labels[i] if i < len(labels) else "").strip()
        parts.append(f"{label}: {text}" if label else text)
    return "\n\n".join(parts)


records = []
for row in labeled:
    abstract = join_sections(row["context"])
    if not abstract:
        continue
    records.append({
        "instruction": f'{TASK}\n\nQuestion: {row["question"].strip()}',
        "input": abstract,
        "output": f'Answer: {row["final_decision"]}\n\n{" ".join(row["long_answer"].split())}',
        "decision": row["final_decision"],
        "pubid": row["pubid"],
    })

# Identical to notebooks 02 and 03, deliberately — same seed, same ordering, same result.
rng = random.Random(SEED + 1)
by_decision = {}
for r in records:
    by_decision.setdefault(r["decision"], []).append(r)

train_records, eval_records = [], []
for decision in sorted(by_decision):
    rows = sorted(by_decision[decision], key=lambda r: r["pubid"])
    rng.shuffle(rows)
    n_eval = round(len(rows) * EVAL_FRACTION)
    eval_records.extend(rows[:n_eval])
    train_records.extend(rows[n_eval:])
rng.shuffle(train_records); rng.shuffle(eval_records)

eval_counts = Counter(r["decision"] for r in eval_records)
MAJORITY_LABEL, majority_n = eval_counts.most_common(1)[0]
MAJORITY_BASELINE = majority_n / len(eval_records)

print(f"eval {len(eval_records)}  {dict(eval_counts)}")
print(f"majority-class baseline: {MAJORITY_BASELINE:.1%} (always answer '{MAJORITY_LABEL}')")

In [ ]:
PROMPT_WITH_INPUT = (
    "Below is an instruction describing a task, paired with input providing further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

PROMPT_NO_INPUT = (
    "Below is an instruction describing a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Response:\n"
)


def build_prompt(record) -> str:
    """The part the model reads but is NOT trained to produce."""
    template = PROMPT_WITH_INPUT if record["input"].strip() else PROMPT_NO_INPUT
    return template.format(instruction=record["instruction"].strip(),
                           input=record["input"].strip())


DECISION_RE = re.compile(r"answer\s*:?\s*\b(yes|no|maybe)\b", re.IGNORECASE)


def parse_decision(text):
    """Notebook 02's grading regex, reused unchanged. None means unparseable."""
    hit = DECISION_RE.search(text)
    return hit.group(1).lower() if hit else None


MAX_PROMPT_LEN = 896        # notebook 02's truncation length
GEN_TOKENS = 120            # notebook 02's generation cap

PROMPT_TOKENS = [len(tokenizer(build_prompt(r), add_special_tokens=False)["input_ids"])
                 for r in eval_records]
print(f"prompt length over the 200 eval records: min {min(PROMPT_TOKENS)}  "
      f"median {int(statistics.median(PROMPT_TOKENS))}  max {max(PROMPT_TOKENS)} tokens")
print(f"a prompt is ~{statistics.median(PROMPT_TOKENS) / GEN_TOKENS:.1f}x longer than its answer, "
      "so prefill is not a rounding error here")

## 7. Baseline: `model.generate()`

The path every fine-tuning tutorial stops at, measured properly so the comparison later is fair.

**Fixed output length.** Every timed call below passes `min_new_tokens == max_new_tokens`, which
suppresses EOS until the cap is reached. That is deliberately *not* how you would run this in
production — it is how you have to run it to measure. Left alone, one system answers in 40 tokens
and the other in 95, and the tokens/sec comparison silently becomes a comparison of verbosity. The
vLLM side uses `ignore_eos` for the same reason.

**Warmup discarded.** The first call pays for cuBLAS autotuning, memory-pool growth and kernel
loading. Including it would slander whichever system got measured first.

**TTFT needs a streamer.** `generate()` returns once, at the end, so there is no first-token
timestamp to read. `TextIteratorStreamer` on a background thread is the only way to get one. One
honest caveat: the streamer decodes incrementally and holds back incomplete UTF-8 sequences, so
what it reports is time-to-first-*decodable-text* — a slight overestimate of true TTFT. The vLLM
side is measured from SSE chunks, which has the same property, so the two stay comparable.

In [ ]:
from threading import Thread
from transformers import TextIteratorStreamer

model.eval()
model.config.use_cache = True


@torch.no_grad()
def hf_generate_natural(batch, max_new_tokens=GEN_TOKENS):
    """Greedy, stopping at EOS — notebook 03's generate_batch, unchanged. For correctness, not timing."""
    enc = tokenizer([build_prompt(r) for r in batch], return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_PROMPT_LEN).to("cuda")
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         eos_token_id=tokenizer.eos_token_id,
                         pad_token_id=tokenizer.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]


@torch.no_grad()
def hf_generate_fixed(batch, max_new_tokens=GEN_TOKENS):
    """Greedy, exactly max_new_tokens per sequence. Also reports how much of the batch was padding."""
    enc = tokenizer([build_prompt(r) for r in batch], return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_PROMPT_LEN).to("cuda")
    pad_fraction = 1.0 - enc["attention_mask"].float().mean().item()
    out = model.generate(**enc, max_new_tokens=max_new_tokens, min_new_tokens=max_new_tokens,
                         do_sample=False, eos_token_id=tokenizer.eos_token_id,
                         pad_token_id=tokenizer.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen], pad_fraction


def hf_stream_once(record, max_new_tokens=GEN_TOKENS):
    """One request, streamed, so TTFT is observable. Returns timings in seconds."""
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    enc = tokenizer(build_prompt(record), return_tensors="pt", truncation=True,
                    max_length=MAX_PROMPT_LEN).to("cuda")
    kwargs = dict(**enc, max_new_tokens=max_new_tokens, min_new_tokens=max_new_tokens,
                  do_sample=False, streamer=streamer,
                  eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    t0 = time.perf_counter()
    thread = Thread(target=model.generate, kwargs=kwargs)
    thread.start()
    ttft, pieces = None, []
    for chunk in streamer:
        if not chunk:
            continue
        if ttft is None:
            ttft = time.perf_counter() - t0
        pieces.append(chunk)
    thread.join()
    total = time.perf_counter() - t0
    return {"ttft": ttft, "total": total, "itl": (total - ttft) / (max_new_tokens - 1),
            "prompt_tokens": enc["input_ids"].shape[1], "text": "".join(pieces)}

In [ ]:
HF_STREAM_N = 20            # requests timed one at a time, for the per-request numbers

for r in eval_records[:2]:                       # warmup, discarded
    hf_stream_once(r)

hf_stream = []
for i, r in enumerate(eval_records[:HF_STREAM_N]):
    hf_stream.append(hf_stream_once(r))
    print(f"\r  streaming: {i + 1}/{HF_STREAM_N}", end="")
print()


def pct(values, q):
    """Percentile by nearest rank — no numpy interpolation games on 20 samples."""
    ordered = sorted(values)
    return ordered[min(len(ordered) - 1, max(0, math.ceil(q / 100 * len(ordered)) - 1))]


HF_TTFT = [h["ttft"] for h in hf_stream]
HF_ITL = [h["itl"] for h in hf_stream]
HF_TOTAL = [h["total"] for h in hf_stream]
HF_DECODE_TOK_S = GEN_TOKENS / statistics.median([h["total"] - h["ttft"] for h in hf_stream])

print(f"\nmodel.generate(), one request at a time, n={HF_STREAM_N}")
print(f"  TTFT      p50 {pct(HF_TTFT, 50) * 1000:7.1f} ms   p95 {pct(HF_TTFT, 95) * 1000:7.1f} ms")
print(f"  ITL       p50 {pct(HF_ITL, 50) * 1000:7.1f} ms   p95 {pct(HF_ITL, 95) * 1000:7.1f} ms")
print(f"  total     p50 {pct(HF_TOTAL, 50):7.2f} s    p95 {pct(HF_TOTAL, 95):7.2f} s")
print(f"  decode    {HF_DECODE_TOK_S:7.1f} tokens/sec")
if ROOFLINE_TOK_S:
    print(f"            {HF_DECODE_TOK_S / ROOFLINE_TOK_S:.0%} of the {ROOFLINE_TOK_S:.0f} tok/s "
          "bandwidth roofline from section 5")

### Static batching, and what it wastes

Batching is how `generate()` gets throughput: one weight read serves the whole batch, so tokens/sec
climbs steeply at first. Two costs come with it, and both are visible below.

**Padding.** Every sequence in a batch is padded to the longest one. Those positions are computed
and then discarded. PubMedQA abstracts vary a lot in length, so the waste is substantial and it
grows with batch size, because a bigger batch is more likely to contain one very long abstract.

**Head-of-line blocking.** The batch returns when its *slowest* member finishes. Here we have
pinned every sequence to exactly `GEN_TOKENS`, which hides that cost entirely — section 14 removes
the pin and measures what it was hiding.

In [ ]:
HF_REPS = 2                 # timed repeats per batch size, after a warmup
HF_BATCH_SIZES = [1, 8, 16, 32]

hf_batch = []
for bs in HF_BATCH_SIZES:
    batch = eval_records[:bs]
    hf_generate_fixed(batch)                     # warmup, discarded
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(HF_REPS):
        _, pad_fraction = hf_generate_fixed(batch)
    torch.cuda.synchronize()
    seconds = (time.perf_counter() - t0) / HF_REPS
    hf_batch.append({"batch": bs, "seconds": seconds, "tok_s": bs * GEN_TOKENS / seconds,
                     "pad_fraction": pad_fraction})
    print(f"\r  batch {bs:2d} done", end="")
print()

print(f"\n{'batch':>6s}{'wall s':>10s}{'tokens/sec':>13s}{'per-request s':>16s}{'padding':>10s}")
print("-" * 55)
for row in hf_batch:
    print(f"{row['batch']:6d}{row['seconds']:10.2f}{row['tok_s']:13.1f}"
          f"{row['seconds']:16.2f}{row['pad_fraction']:10.1%}")

HF_PEAK_TOK_S = max(r["tok_s"] for r in hf_batch)
print(f"\npeak {HF_PEAK_TOK_S:.1f} tokens/sec at batch "
      f"{max(hf_batch, key=lambda r: r['tok_s'])['batch']}")
print("Note the per-request column: batching buys throughput by making every individual request")
print("wait longer. That trade is the whole subject of section 13.")

### And the numbers section 11 compares against

One full pass over the 200 held-out records through `model.generate()`, **stopping at EOS** rather
than at a fixed length. This is the notebook-03 scoring path, unchanged, and it is what section 11
holds the server to. Roughly two minutes.

The fixed-length runs above and this one measure different things on purpose: pin the length when
timing, let the model stop when scoring. Mixing those up is how a benchmark ends up comparing a
system that emitted 40 tokens against one that emitted 120.

In [ ]:
HF_SCORE_BATCH = 16

hf_all_texts = []
for i in range(0, len(eval_records), HF_SCORE_BATCH):
    hf_all_texts.extend(hf_generate_natural(eval_records[i:i + HF_SCORE_BATCH]))
    print(f"\r  scoring: {min(i + HF_SCORE_BATCH, len(eval_records))}/{len(eval_records)}", end="")
print()

HF_TEXTS = hf_all_texts
HF_PREDS = [parse_decision(t) for t in HF_TEXTS]
HF_ACCURACY = sum(p == r["decision"] for p, r in zip(HF_PREDS, eval_records)) / len(eval_records)
HF_PARSE_RATE = sum(p is not None for p in HF_PREDS) / len(eval_records)
HF_MEAN_TOKENS = statistics.mean(
    len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in HF_TEXTS)

print(f"\nmodel.generate() over the same 200 records")
print(f"  accuracy    {HF_ACCURACY:.1%}   (majority-class baseline {MAJORITY_BASELINE:.1%})")
print(f"  parse rate  {HF_PARSE_RATE:.1%}")
print(f"  mean output {HF_MEAN_TOKENS:.0f} tokens, against a {GEN_TOKENS}-token cap — the model is")
print(f"              stopping on its own, which is what notebook 02 installed")

### One more measurement, while the model is still loaded

Section 14 compares static batching against vLLM's continuous batching, and the static half has to
be measured here — by the time that section runs, these weights are gone and the GPU belongs to the
server.

The workload is deliberately **ragged**: 48 requests whose output lengths are drawn from
24 to 256 tokens. Every benchmark above pinned all sequences to the same length, which is the one
condition under which static batching looks fine. Real traffic is never that tidy.

A static batch runs until its **longest** member finishes. The short requests in it are done early
and then occupy a slot, computing padding, until the batch retires. The cell below measures both
the wall clock and how much of the compute was for tokens anybody asked for.

In [ ]:
# Fixed and seeded, so section 14 sends the identical workload to the server.
VARIED_N = 48
VARIED_BATCH = 8
_len_rng = random.Random(SEED)
VARIED_LENGTHS = [_len_rng.choice([24, 48, 96, 160, 256]) for _ in range(VARIED_N)]
VARIED_RECORDS = [eval_records[i % len(eval_records)] for i in range(VARIED_N)]

hf_generate_fixed(VARIED_RECORDS[:VARIED_BATCH], max_new_tokens=32)      # warmup, discarded

useful_tokens = sum(VARIED_LENGTHS)
issued_tokens = 0
torch.cuda.synchronize()
t0 = time.perf_counter()
for i in range(0, VARIED_N, VARIED_BATCH):
    batch = VARIED_RECORDS[i:i + VARIED_BATCH]
    lengths = VARIED_LENGTHS[i:i + VARIED_BATCH]
    longest = max(lengths)                       # the batch runs until its slowest member is done
    issued_tokens += longest * len(batch)
    hf_generate_fixed(batch, max_new_tokens=longest)
    print(f"\r  static batches: {i // VARIED_BATCH + 1}/{VARIED_N // VARIED_BATCH}", end="")
torch.cuda.synchronize()
HF_VARIED_WALL = time.perf_counter() - t0
HF_VARIED_EFFICIENCY = useful_tokens / issued_tokens
print()

print(f"\nstatic batching, {VARIED_N} ragged requests at batch {VARIED_BATCH}")
print(f"  wall clock          {HF_VARIED_WALL:.1f} s")
print(f"  tokens requested    {useful_tokens:,}")
print(f"  tokens computed     {issued_tokens:,}")
print(f"  useful fraction     {HF_VARIED_EFFICIENCY:.1%}  <- the rest is head-of-line blocking")
print(f"  effective rate      {useful_tokens / HF_VARIED_WALL:.1f} useful tokens/sec")

## 8. Free the GPU completely

vLLM claims `gpu_memory_utilization` of **total** VRAM, not of free VRAM. A forgotten 2.5 GB model
does not cause an error — it causes vLLM to reserve a much smaller KV cache, which caps concurrency,
which shows up in section 13 as poor throughput that has nothing to do with vLLM.

This is the single most common way a home-grown serving benchmark ends up wrong, so it gets an
assertion rather than a comment.

In [ ]:
before_gb = torch.cuda.memory_allocated() / 1e9

del model
gc.collect()
torch.cuda.empty_cache()

after_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9
print(f"allocated {before_gb:.2f} GB -> {after_gb:.2f} GB   (reserved by the caching allocator: "
      f"{reserved_gb:.2f} GB)")

assert after_gb < 0.5, (
    f"{after_gb:.2f} GB is still allocated. vllm reserves a fraction of TOTAL memory, so it will "
    "start anyway with a smaller KV cache and every throughput number below will be measuring the "
    "leftovers rather than the engine. Find the reference that is still live before continuing."
)
print("\nGPU is clear. The server starts in its own process, so torch's allocator here is empty.")

## 9. `vllm serve`

An HTTP server, in a separate process, speaking the OpenAI API. That last part is the reason this
step is worth taking: no client code in the world needs to know that the model behind the endpoint
is a 1B Llama with three LoRAs merged into it.

Every flag is passed explicitly, including the ones whose defaults would have been fine, because a
launch command is the most durable documentation a deployment has:

| flag | why it is here |
|---|---|
| `--dtype bfloat16` | Section 4, trap 2. Never inherit this from the config. |
| `--max-model-len 1280` | Section 4, trap 1. Without it the server will not start. |
| `--gpu-memory-utilization 0.85` | Fraction of **total** VRAM. Sets the KV cache, and so the concurrency ceiling. |
| `--max-num-seqs 256` | Hard cap on in-flight sequences. Bounds tail latency; section 13 shows it binding. |
| `--served-model-name` | Decouples the name clients use from the path on disk. Redeploying is then a path change, not a client change. |
| `--port 8000` | |

Two more things about this cell. It **blocks on `/health`** rather than sleeping a fixed number of
seconds, and it **prints the log tail if the process dies**, because a server that exits silently
during startup in a Colab background process is otherwise nearly impossible to debug.

In [ ]:
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}"
SERVED_NAME = "pubmedqa-1b"
LOG_PATH = Path("/content/outputs/vllm-server.log")

SERVER = None       # the Popen handle; sections 15 and 16 restart the server, so it lives here


def stop_server():
    """Terminate the running server and give the CUDA context time to actually go away."""
    global SERVER
    if SERVER is None:
        return
    SERVER.terminate()
    try:
        SERVER.wait(timeout=90)
    except subprocess.TimeoutExpired:
        SERVER.kill()
        SERVER.wait(timeout=30)
    SERVER = None
    time.sleep(8)


def start_server(model_path, extra_args=(), timeout=900):
    """Launch `vllm serve` in the background, block until /health answers, tail the log if it dies."""
    global SERVER
    stop_server()
    cmd = ["vllm", "serve", str(model_path),
           "--served-model-name", SERVED_NAME,
           "--dtype", "bfloat16",
           "--max-model-len", str(MAX_MODEL_LEN),
           "--gpu-memory-utilization", str(GPU_MEM_UTIL),
           "--max-num-seqs", str(MAX_NUM_SEQS),
           "--port", str(PORT),
           *extra_args]
    print(" ".join(cmd) + "\n")
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    SERVER = subprocess.Popen(cmd, stdout=LOG_PATH.open("w"), stderr=subprocess.STDOUT)

    t0 = time.perf_counter()
    while time.perf_counter() - t0 < timeout:
        if SERVER.poll() is not None:
            print("\n" + LOG_PATH.read_text()[-3000:])
            raise RuntimeError(
                f"vllm serve exited with code {SERVER.returncode} during startup. The tail of its "
                "log is above — the first traceback in it is the real error."
            )
        try:
            with urllib.request.urlopen(f"{BASE_URL}/health", timeout=2) as resp:
                if resp.status == 200:
                    print(f"\rready in {time.perf_counter() - t0:.0f}s" + " " * 20)
                    return SERVER
        except Exception:
            pass
        print(f"\r  waiting for /health ... {time.perf_counter() - t0:.0f}s", end="")
        time.sleep(3)

    print("\n" + LOG_PATH.read_text()[-3000:])
    stop_server()
    raise TimeoutError(f"Server was not ready within {timeout}s. Log tail above.")


start_server(EXPORT_DIR)

In [ ]:
def http_json(path, payload=None, timeout=600):
    """Minimal JSON client. The point of this section is the endpoint, not the library."""
    url = f"{BASE_URL}{path}"
    if payload is None:
        request = urllib.request.Request(url)
    else:
        request = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                         headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as resp:
        return json.loads(resp.read())


models = http_json("/v1/models")
print("GET /v1/models")
for m in models["data"]:
    print(f"  id={m['id']!r}  owned_by={m.get('owned_by')}  max_model_len={m.get('max_model_len')}")

reply = http_json("/v1/completions", {"model": SERVED_NAME,
                                      "prompt": build_prompt(eval_records[0]),
                                      "max_tokens": GEN_TOKENS, "temperature": 0.0})
print("\nPOST /v1/completions")
print(f"  finish_reason  {reply['choices'][0]['finish_reason']}")
print(f"  usage          {reply['usage']}")
print(f"  text           {' '.join(reply['choices'][0]['text'].split())[:300]}")
print(f"\n  gold decision  {eval_records[0]['decision']}")

### The chat endpoint, as promised

Section 4 said `/v1/chat/completions` cannot work on a base model with no chat template. Rather
than assert that, ask the server.

This is not a vLLM limitation, and adding a chat template to the tokenizer would not fix it in any
way that helps — notebooks 02 and 03 trained on the `### Instruction: / ### Response:` template, so
a chat template would have to reproduce exactly that string to produce a prompt the model
recognises. At which point you have re-implemented `build_prompt` in Jinja. Serving a base model
through `/v1/completions` and formatting on the client is the honest arrangement.

In [ ]:
try:
    http_json("/v1/chat/completions", {"model": SERVED_NAME,
                                       "messages": [{"role": "user", "content": "hello"}]})
    print("Unexpected: the chat endpoint answered. Something added a chat template.")
except urllib.error.HTTPError as exc:
    body = exc.read().decode()
    print(f"HTTP {exc.code} from /v1/chat/completions, as expected")
    print(f"  {' '.join(body.split())[:300]}")

### Any OpenAI client, unchanged

The reason to serve an OpenAI-compatible API rather than a bespoke Flask route: every SDK, gateway,
proxy, evaluation harness and observability tool that already speaks it now speaks to this model.
Two lines change — `base_url` and a dummy `api_key`.

In [ ]:
from openai import OpenAI       # ships as a vllm dependency; no extra install

client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="EMPTY")   # the server does not check it
completion = client.completions.create(
    model=SERVED_NAME,
    prompt=build_prompt(eval_records[1]),
    max_tokens=GEN_TOKENS,
    temperature=0.0,
)
print(" ".join(completion.choices[0].text.split())[:320])
print(f"\ngold decision: {eval_records[1]['decision']}")

## 10. Read the server's own accounting

Section 5 predicted a KV cache size and a concurrency ceiling from the config file. vLLM computes
the same quantities properly at startup — it profiles a real forward pass rather than guessing at
activation overhead — and prints them. Comparing the two is free, and a large gap means the mental
model is wrong somewhere worth finding.

In [ ]:
log_text = LOG_PATH.read_text()

kv_match = re.search(r"KV cache size:\s*([\d,]+)\s*tokens", log_text)
conc_match = re.search(r"Maximum concurrency for ([\d,]+) tokens per request:\s*([\d.]+)x", log_text)

print(f"predicted (section 5)   {PREDICTED_KV_TOKENS:>12,.0f} tokens of KV cache")
if kv_match:
    actual_tokens = int(kv_match.group(1).replace(",", ""))
    print(f"reported by vllm        {actual_tokens:>12,d} tokens")
    print(f"ratio                   {actual_tokens / PREDICTED_KV_TOKENS:>12.2f}x")
else:
    actual_tokens = None
    print("vllm did not print a KV cache line in the format this cell expects.")

print(f"\npredicted concurrency   {PREDICTED_SEQS:>12,.0f} sequences at "
      f"max_model_len={MAX_MODEL_LEN}")
if conc_match:
    print(f"reported by vllm        {float(conc_match.group(2)):>12,.1f} sequences at "
          f"{conc_match.group(1)} tokens each")
print(f"--max-num-seqs          {MAX_NUM_SEQS:>12,d}  <- whichever of these two is smaller is the "
      "real ceiling")

# Print the raw lines too, so a change in vllm's log format costs you nothing.
print("\nthe engine's own words:")
for line in log_text.splitlines():
    if any(k in line for k in ("KV cache size", "Maximum concurrency", "GPU blocks",
                               "memory profiling", "Available KV cache memory")):
        print("  " + line.strip()[-160:])

## 11. Correctness before speed

Everything after this section is a performance number, and a performance number for a model that
answers differently is worthless. So: the same 200 held-out articles, the same grading regex,
through the server.

Two things get checked, and they are not the same thing:

**Accuracy parity.** Does the served model score what notebook 03 measured? This is the one that
matters, and it should hold to within a point.

**Per-record agreement.** Does it give the *same answer* to each individual article? This will be
high and it will not be 100%, and the reason is worth internalising: greedy decoding is
deterministic given identical arithmetic, and two engines do not do identical arithmetic. Different
kernels, different batch shapes, different reduction orders in bf16 — wherever the top two logits
are close, the argmax can flip. **Greedy is not bit-reproducible across engines**, and a serving
migration that expects it to be will fail a naive regression test on every deploy.

The right regression test is the one below: assert on the *metric*, report the agreement rate, and
investigate only if agreement collapses.

In [ ]:
import aiohttp


def run_async(coro):
    """asyncio.run() raises inside Colab's already-running loop; nest_asyncio makes it work."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import nest_asyncio
    nest_asyncio.apply()
    return asyncio.get_event_loop().run_until_complete(coro)


async def one_request(session, prompt, max_tokens=GEN_TOKENS, ignore_eos=False,
                      served=None, stream=True):
    """One /v1/completions call. With stream=True the first-token time is observable."""
    payload = {"model": served or SERVED_NAME, "prompt": prompt, "max_tokens": max_tokens,
               "temperature": 0.0, "ignore_eos": ignore_eos, "stream": stream}
    t0 = time.perf_counter()
    async with session.post(f"{BASE_URL}/v1/completions", json=payload) as resp:
        resp.raise_for_status()
        if not stream:
            body = await resp.json()
            return {"ttft": None, "itl": None, "total": time.perf_counter() - t0,
                    "text": body["choices"][0]["text"],
                    "output_tokens": body["usage"]["completion_tokens"]}
        ttft, pieces = None, []
        async for raw in resp.content:
            line = raw.decode("utf-8").strip()
            if not line.startswith("data:"):
                continue
            chunk = line[5:].strip()
            if chunk == "[DONE]":
                break
            choices = json.loads(chunk).get("choices") or []
            if choices and choices[0].get("text"):
                if ttft is None:
                    ttft = time.perf_counter() - t0
                pieces.append(choices[0]["text"])
    total = time.perf_counter() - t0
    n = len(pieces)
    return {"ttft": ttft, "total": total, "text": "".join(pieces), "output_tokens": n,
            "itl": (total - ttft) / max(1, n - 1) if ttft is not None else None}


async def run_load(prompts, concurrency, **kw):
    """Send every prompt, at most `concurrency` in flight. Returns (results, wall_seconds)."""
    sem = asyncio.Semaphore(concurrency)

    async def guarded(session, prompt):
        async with sem:
            return await one_request(session, prompt, **kw)

    connector = aiohttp.TCPConnector(limit=concurrency + 8)
    async with aiohttp.ClientSession(connector=connector,
                                     timeout=aiohttp.ClientTimeout(total=3600)) as session:
        t0 = time.perf_counter()
        results = await asyncio.gather(*(guarded(session, p) for p in prompts))
        return results, time.perf_counter() - t0

In [ ]:
EVAL_PROMPTS = [build_prompt(r) for r in eval_records]

served_results, served_wall = run_async(run_load(EVAL_PROMPTS, concurrency=16, stream=False))
served_preds = [parse_decision(r["text"]) for r in served_results]

gold = [r["decision"] for r in eval_records]
vllm_accuracy = sum(p == g for p, g in zip(served_preds, gold)) / len(gold)
vllm_parse_rate = sum(p is not None for p in served_preds) / len(gold)

print(f"scored {len(eval_records)} records in {served_wall:.1f}s at concurrency 16")
print(f"\n{'system':34s}{'accuracy':>10s}{'parse rate':>13s}")
print("-" * 57)
print(f"{'majority class (always ' + MAJORITY_LABEL + ')':34s}{MAJORITY_BASELINE:>10.1%}{'n/a':>13s}")
print(f"{'model.generate() (section 7)':34s}{HF_ACCURACY:>10.1%}{HF_PARSE_RATE:>13.1%}")
print(f"{'vllm serve, merged checkpoint':34s}{vllm_accuracy:>10.1%}{vllm_parse_rate:>13.1%}")
print(f"\naccuracy delta between the two paths: {abs(vllm_accuracy - HF_ACCURACY):.1%}")

In [ ]:
# Both sides now have a natural-EOS greedy pass over the same 200 records, so they compare.
agree = sum(a == b for a, b in zip(HF_PREDS, served_preds))
identical = sum(1 for h, s_ in zip(HF_TEXTS, served_results)
                if " ".join(h.split()) == " ".join(s_["text"].split()))

print(f"over all {len(eval_records)} records:")
print(f"  same parsed decision   {agree}/{len(eval_records)}  ({agree / len(eval_records):.0%})")
print(f"  byte-identical text    {identical}/{len(eval_records)}  ({identical / len(eval_records):.0%})")
print("\nThe second number is the honest one. Two engines running 'the same' greedy decode do not")
print("do the same arithmetic, and near-ties in the logits break differently. Regression-test the")
print("metric, not the string.")

assert abs(vllm_accuracy - HF_ACCURACY) < 0.05, (
    f"The server scores {vllm_accuracy:.1%} against {HF_ACCURACY:.1%} through model.generate(). "
    "More than five points apart is not kernel non-determinism — it means the served checkpoint, "
    "the prompt template or the sampling parameters differ from what notebook 03 measured. "
    "Nothing below is worth reading until this matches."
)
assert vllm_parse_rate > 0.90, (
    f"Parse rate collapsed to {vllm_parse_rate:.1%}. The served model is not producing the "
    "'Answer: yes/no/maybe' format notebook 02 trained, so the prompt reaching it is not the "
    "prompt it was trained on."
)

## 12. Latency: TTFT, ITL, and the tail

Three numbers describe a streaming request, and they are bounded by different hardware:

| | what it is | what limits it |
|---|---|---|
| **TTFT** | time to first token | **prefill** — one forward pass over the whole prompt. Compute-bound, scales with prompt length. |
| **ITL** | inter-token latency, first token to last | **decode** — one forward pass per token, reading every weight each time. Memory-bandwidth-bound, roughly flat in prompt length. |
| **end-to-end** | TTFT + ITL x tokens | both |

Which one you care about is a product decision, not a technical one. A chat UI lives or dies on
TTFT, because that is when the user stops looking at a spinner. A batch summarisation job does not
care about TTFT at all and only wants tokens/sec. Reporting a single "latency" number hides the
distinction and is how teams optimise the wrong half.

`ignore_eos=True` here pins every response to exactly `GEN_TOKENS`, matching section 7's
fixed-length runs. Concurrency is 1, so this is the clean per-request cost with no queueing in it.

In [ ]:
LATENCY_N = 40

lat_results, lat_wall = run_async(
    run_load(EVAL_PROMPTS[:LATENCY_N], concurrency=1, ignore_eos=True, stream=True))

# With ignore_eos the token count is known exactly, so ITL does not have to trust the SSE chunk
# count (vllm may pack more than one token into a chunk).
for r in lat_results:
    r["itl"] = (r["total"] - r["ttft"]) / (GEN_TOKENS - 1)

VLLM_TTFT = [r["ttft"] for r in lat_results]
VLLM_ITL = [r["itl"] for r in lat_results]
VLLM_TOTAL = [r["total"] for r in lat_results]
VLLM_DECODE_TOK_S = GEN_TOKENS / statistics.median([r["total"] - r["ttft"] for r in lat_results])

print(f"{'':22s}{'vllm serve':>14s}{'model.generate()':>20s}")
print("-" * 56)
for name, v, h in [("TTFT p50 (ms)", pct(VLLM_TTFT, 50) * 1000, pct(HF_TTFT, 50) * 1000),
                   ("TTFT p95 (ms)", pct(VLLM_TTFT, 95) * 1000, pct(HF_TTFT, 95) * 1000),
                   ("ITL  p50 (ms)", pct(VLLM_ITL, 50) * 1000, pct(HF_ITL, 50) * 1000),
                   ("ITL  p95 (ms)", pct(VLLM_ITL, 95) * 1000, pct(HF_ITL, 95) * 1000),
                   ("total p50 (s)", pct(VLLM_TOTAL, 50), pct(HF_TOTAL, 50)),
                   ("total p95 (s)", pct(VLLM_TOTAL, 95), pct(HF_TOTAL, 95))]:
    print(f"{name:22s}{v:>14.1f}{h:>20.1f}")

print(f"\ndecode rate at concurrency 1: {VLLM_DECODE_TOK_S:.1f} tokens/sec")
if ROOFLINE_TOK_S:
    print(f"  {VLLM_DECODE_TOK_S / ROOFLINE_TOK_S:.0%} of the {ROOFLINE_TOK_S:.0f} tok/s bandwidth "
          "roofline from section 5")
    assert VLLM_DECODE_TOK_S < ROOFLINE_TOK_S * 1.2, (
        f"Measured {VLLM_DECODE_TOK_S:.0f} tok/s against a {ROOFLINE_TOK_S:.0f} tok/s bandwidth "
        "ceiling. A single stream cannot decode faster than the weights can be read, so the "
        "measurement is wrong — most likely the responses are shorter than GEN_TOKENS."
    )
print(f"\nn={LATENCY_N}. p95 is the last-but-two sample here; p99 would be the maximum, which is a")
print("property of this sample rather than of the server. Reporting one would be dishonest.")

### TTFT is prefill, and prefill is not free

The 200 held-out prompts vary by several hundred tokens, which is enough to see the two regimes
separate. TTFT should climb with prompt length; ITL should not care.

This is why prompt engineering has a latency cost that nobody budgets for. Adding 500 tokens of
few-shot examples to a system prompt does not slow generation down at all — it moves TTFT, which is
exactly the number the user experiences as "is it broken?".

In [ ]:
lat_prompt_tokens = PROMPT_TOKENS[:LATENCY_N]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].scatter(lat_prompt_tokens, [t * 1000 for t in VLLM_TTFT], s=22, color="#4C72B0")
axes[0].set_xlabel("prompt tokens")
axes[0].set_ylabel("TTFT (ms)")
axes[0].set_title("Time to first token is prefill")
axes[0].grid(alpha=0.3)

axes[1].scatter(lat_prompt_tokens, [t * 1000 for t in VLLM_ITL], s=22, color="#C44E52")
axes[1].set_xlabel("prompt tokens")
axes[1].set_ylabel("inter-token latency (ms)")
axes[1].set_title("Decode does not care how long the prompt was")
axes[1].set_ylim(0, max(VLLM_ITL) * 1000 * 1.6)
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Correlation, so the eye is not the only judge.
def corr(xs, ys):
    """Pearson r, written out rather than pulling in scipy for one number."""
    mx, my = statistics.mean(xs), statistics.mean(ys)
    num = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    den = math.sqrt(sum((x - mx) ** 2 for x in xs) * sum((y - my) ** 2 for y in ys))
    return num / den if den else float("nan")


print(f"correlation with prompt length:  TTFT r={corr(lat_prompt_tokens, VLLM_TTFT):+.2f}   "
      f"ITL r={corr(lat_prompt_tokens, VLLM_ITL):+.2f}")

## 13. Throughput against concurrency

The measurement that decides how many GPUs you buy.

One request at a time leaves the GPU almost idle: decoding a token reads all 2.5 GB of weights to
produce a single token per sequence. Run 32 sequences and the same weight read serves all 32. This
is why throughput climbs steeply at first and why serving a model one request at a time is the most
expensive way to do it.

It does not climb forever. Eventually either the KV cache fills, `--max-num-seqs` binds, or the GPU
saturates — and past that point additional concurrency only adds queueing, so throughput flattens
while latency keeps rising. **The knee of that curve is the operating point**, and finding it is the
whole purpose of this section.

One caveat about the method, stated because it is a real limitation: this is a **closed-loop** load
generator. It keeps exactly N requests in flight and only sends a new one when an old one finishes,
so it can never build an unbounded queue. Real traffic arrives whether or not you are ready for it,
and an open-loop test at a fixed arrival rate is the one that shows queueing collapse. The frontier
below is honest about the machine and optimistic about the world.

In [ ]:
CONCURRENCY_LEVELS = [1, 2, 4, 8, 16, 32, 64]
LOAD_N = 64                 # requests per level

pool = (EVAL_PROMPTS * ((LOAD_N // len(EVAL_PROMPTS)) + 1))[:LOAD_N]

sweep = []
for level in CONCURRENCY_LEVELS:
    results, wall = run_async(run_load(pool, concurrency=level, ignore_eos=True, stream=True))
    ttfts = [r["ttft"] for r in results]
    totals = [r["total"] for r in results]
    output_tokens = LOAD_N * GEN_TOKENS          # exact, because ignore_eos pins the length
    sweep.append({
        "concurrency": level,
        "tok_s": output_tokens / wall,
        "req_s": LOAD_N / wall,
        "ttft_p50": pct(ttfts, 50), "ttft_p95": pct(ttfts, 95),
        "e2e_p50": pct(totals, 50), "e2e_p95": pct(totals, 95),
    })
    print(f"\r  concurrency {level:3d} done", end="")
print()

print(f"\n{'conc':>5s}{'tok/s':>10s}{'req/s':>8s}{'TTFT p50':>11s}{'TTFT p95':>11s}"
      f"{'e2e p50':>10s}{'e2e p95':>10s}")
print("-" * 65)
for row in sweep:
    print(f"{row['concurrency']:5d}{row['tok_s']:10.0f}{row['req_s']:8.2f}"
          f"{row['ttft_p50'] * 1000:10.0f}m{row['ttft_p95'] * 1000:10.0f}m"
          f"{row['e2e_p50']:9.2f}s{row['e2e_p95']:9.2f}s")

VLLM_PEAK = max(sweep, key=lambda r: r["tok_s"])
print(f"\npeak {VLLM_PEAK['tok_s']:.0f} tokens/sec at concurrency {VLLM_PEAK['concurrency']}")
print(f"  {VLLM_PEAK['tok_s'] / sweep[0]['tok_s']:.1f}x the single-stream rate, on the same GPU, "
      "for the same weights")
print(f"  and p95 end-to-end went from {sweep[0]['e2e_p95']:.2f}s to "
      f"{VLLM_PEAK['e2e_p95']:.2f}s to get there")

In [ ]:
conc = [r["concurrency"] for r in sweep]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].plot(conc, [r["tok_s"] for r in sweep], "o-", color="#4C72B0")
if ROOFLINE_TOK_S:
    axes[0].axhline(ROOFLINE_TOK_S, color="#888888", ls="--",
                    label=f"batch-1 roofline ({ROOFLINE_TOK_S:.0f})")
    axes[0].legend(fontsize=8)
axes[0].set_xscale("log", base=2); axes[0].set_xlabel("concurrent requests")
axes[0].set_ylabel("output tokens/sec"); axes[0].set_title("Throughput")
axes[0].grid(alpha=0.3)

axes[1].plot(conc, [r["ttft_p95"] * 1000 for r in sweep], "o-", color="#C44E52", label="TTFT p95")
axes[1].plot(conc, [r["ttft_p50"] * 1000 for r in sweep], "o--", color="#C44E52", alpha=0.5,
             label="TTFT p50")
axes[1].set_xscale("log", base=2); axes[1].set_xlabel("concurrent requests")
axes[1].set_ylabel("ms"); axes[1].set_title("Queueing shows up in TTFT first")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].plot([r["tok_s"] for r in sweep], [r["e2e_p95"] for r in sweep], "o-", color="#55A868")
for r in sweep:
    axes[2].annotate(str(r["concurrency"]), (r["tok_s"], r["e2e_p95"]),
                     textcoords="offset points", xytext=(5, 4), fontsize=8)
axes[2].set_xlabel("output tokens/sec"); axes[2].set_ylabel("p95 end-to-end (s)")
axes[2].set_title("The frontier — pick a point on this curve")
axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print("The third panel is the one to keep. Every point is an operating choice: move right for")
print("cheaper tokens, move down for happier users. A latency SLO is a horizontal line across it,")
print("and the throughput where it crosses the curve is what one GPU is actually worth to you.")

## 14. Continuous batching, measured

Everything so far pinned every response to the same length, which is the single condition under
which static batching is not embarrassing. Section 7 ran 48 ragged requests — output lengths from
24 to 256 tokens — through `model.generate()` in static batches of 8 and recorded how much of the
compute went to tokens nobody asked for.

The mechanism, in one line each:

- **Static batching** picks 8 requests, runs them together, and returns when the *last* one
  finishes. A request that wanted 24 tokens sits in the batch for 256 steps, computing padding.
- **Continuous batching** schedules per token. A finished sequence leaves at the next step and a
  queued one takes its slot immediately. Nothing waits for anything else to finish.

Same 48 requests, same lengths, same GPU.

In [ ]:
async def run_varied(concurrency):
    """The section-7 ragged workload, each request asking for its own number of tokens."""
    sem = asyncio.Semaphore(concurrency)

    async def guarded(session, record, n_tokens):
        async with sem:
            return await one_request(session, build_prompt(record), max_tokens=n_tokens,
                                     ignore_eos=True, stream=True)

    connector = aiohttp.TCPConnector(limit=concurrency + 8)
    async with aiohttp.ClientSession(connector=connector,
                                     timeout=aiohttp.ClientTimeout(total=3600)) as session:
        t0 = time.perf_counter()
        out = await asyncio.gather(*(guarded(session, r, n)
                                     for r, n in zip(VARIED_RECORDS, VARIED_LENGTHS)))
        return out, time.perf_counter() - t0


varied_results, VLLM_VARIED_WALL = run_async(run_varied(VARIED_BATCH))
useful_tokens = sum(VARIED_LENGTHS)

print(f"{VARIED_N} ragged requests, output lengths {min(VARIED_LENGTHS)}-{max(VARIED_LENGTHS)} "
      f"tokens, {VARIED_BATCH} at a time\n")
print(f"{'':28s}{'static (generate)':>20s}{'continuous (vllm)':>20s}")
print("-" * 68)
print(f"{'wall clock (s)':28s}{HF_VARIED_WALL:>20.1f}{VLLM_VARIED_WALL:>20.1f}")
print(f"{'useful tokens/sec':28s}{useful_tokens / HF_VARIED_WALL:>20.1f}"
      f"{useful_tokens / VLLM_VARIED_WALL:>20.1f}")
print(f"{'compute spent on real tokens':28s}{HF_VARIED_EFFICIENCY:>20.1%}{'~100%':>20s}")
print(f"\nspeedup on this workload: {HF_VARIED_WALL / VLLM_VARIED_WALL:.1f}x")
print(f"of which {1 / HF_VARIED_EFFICIENCY:.1f}x is explained by static batching computing padding,")
print("and the rest by faster kernels and PagedAttention not fragmenting the KV cache.")

## 15. Prefix caching

Every prompt in this workload begins with the same thing: the template header, then the task
instruction from notebook 02. Only after that does it diverge into a question and an abstract.
Those shared tokens get their KV entries recomputed on every single request, forever, for no reason.

`--enable-prefix-caching` keeps KV blocks around and reuses them when a new request starts with
tokens already in the cache. It is close to free — PagedAttention already stores the cache in
shareable blocks, so this is mostly a matter of not throwing them away.

Two measurements, because the honest answer depends entirely on the workload:

1. **This workload.** The shared prefix is short relative to a ~700-token abstract, so expect a
   modest improvement.
2. **A repeated prompt.** The whole prefix hits. This is the shape of a long system prompt, a
   few-shot block, or a RAG context reused across a conversation — the case where prefix caching
   stops being a micro-optimisation.

In [ ]:
def common_prefix_tokens(prompts, sample=60):
    """How many leading tokens every prompt in the sample shares."""
    ids = [tokenizer(p, add_special_tokens=False)["input_ids"] for p in prompts[:sample]]
    n = 0
    for position in zip(*ids):
        if len(set(position)) != 1:
            break
        n += 1
    return n


SHARED_PREFIX = common_prefix_tokens(EVAL_PROMPTS)
print(f"shared prefix across the eval prompts: {SHARED_PREFIX} tokens")
print(f"median prompt: {int(statistics.median(PROMPT_TOKENS))} tokens")
print(f"so at best prefix caching removes {SHARED_PREFIX / statistics.median(PROMPT_TOKENS):.0%} "
      "of prefill on this workload")

start_server(EXPORT_DIR, extra_args=["--enable-prefix-caching"])

In [ ]:
# 1. The real workload, same 40 prompts as section 12.
cached_results, _ = run_async(
    run_load(EVAL_PROMPTS[:LATENCY_N], concurrency=1, ignore_eos=True, stream=True))
CACHED_TTFT = [r["ttft"] for r in cached_results]

# 2. The same prompt, twice: a cold pass to populate the cache, then a warm one.
repeat_prompt = [EVAL_PROMPTS[0]] * 8
run_async(run_load(repeat_prompt, concurrency=1, max_tokens=8, ignore_eos=True, stream=True))
warm, _ = run_async(run_load(repeat_prompt, concurrency=1, max_tokens=8, ignore_eos=True,
                             stream=True))
WARM_TTFT = [r["ttft"] for r in warm]

print(f"{'':30s}{'TTFT p50 (ms)':>16s}{'TTFT p95 (ms)':>16s}")
print("-" * 62)
print(f"{'no prefix caching':30s}{pct(VLLM_TTFT, 50) * 1000:>16.1f}{pct(VLLM_TTFT, 95) * 1000:>16.1f}")
print(f"{'prefix caching, real prompts':30s}{pct(CACHED_TTFT, 50) * 1000:>16.1f}"
      f"{pct(CACHED_TTFT, 95) * 1000:>16.1f}")
print(f"{'prefix caching, repeated prompt':30s}{pct(WARM_TTFT, 50) * 1000:>16.1f}"
      f"{pct(WARM_TTFT, 95) * 1000:>16.1f}")

print(f"\nreal prompts:     {1 - pct(CACHED_TTFT, 50) / pct(VLLM_TTFT, 50):+.0%} on TTFT p50")
print(f"repeated prompt:  {1 - pct(WARM_TTFT, 50) / pct(VLLM_TTFT, 50):+.0%} on TTFT p50")
print("\nThe second row is what this workload is worth. The third is what prefix caching is worth")
print("when the shared prefix is most of the prompt, which is the case worth designing for: put")
print("the stable part of a prompt FIRST and the variable part last, and this becomes free speed.")

## 16. Merged, or hot-swapped?

Section 3 merged all three adapters into one checkpoint. There is another way to serve exactly the
same model: keep the base weights loaded once and attach the LoRA at request time.
`--enable-lora` does that, and it changes the economics of serving fine-tuned models completely.

- **Merged**: one 2.5 GB checkpoint per fine-tune. Ten customers, ten checkpoints, and no GPU holds
  ten of them.
- **Hot-swapped**: one copy of the base weights plus a 45 MB adapter per customer. Ten customers on
  one GPU, and adding the eleventh is a file copy.

The catch is the one section 3 set this up for: **an adapter is only valid against the weights it
was trained on.** Stage 3 was trained on base + stage 1 + stage 2 merged, so that is what has to be
loaded underneath it — which is exactly why `serving-base-s12-bf16` was written on the way past.
Pointing this at `unsloth/Llama-3.2-1B` would load the adapter without error and serve a model that
has never existed.

So this section checks correctness first and speed second, again.

In [ ]:
start_server(LORA_BASE_DIR, extra_args=["--enable-lora",
                                        "--lora-modules", f"stage3={STAGE3_DIR}",
                                        "--max-lora-rank", "16"])

print("\nGET /v1/models")
for m in http_json("/v1/models")["data"]:
    print(f"  {m['id']!r}   <- {'the LoRA' if m['id'] == 'stage3' else 'the base underneath it'}")

In [ ]:
# Correctness first: base+stage3-as-adapter must agree with the fully merged checkpoint.
lora_results, lora_wall = run_async(
    run_load(EVAL_PROMPTS, concurrency=16, stream=False, served="stage3"))
lora_preds = [parse_decision(r["text"]) for r in lora_results]
lora_accuracy = sum(p == g for p, g in zip(lora_preds, gold)) / len(gold)
lora_agree = sum(a == b for a, b in zip(lora_preds, served_preds)) / len(gold)

print(f"{'system':38s}{'accuracy':>10s}{'agrees with merged':>21s}")
print("-" * 69)
print(f"{'merged checkpoint (section 11)':38s}{vllm_accuracy:>10.1%}{'-':>21s}")
print(f"{'base(1+2) + stage-3 LoRA at runtime':38s}{lora_accuracy:>10.1%}{lora_agree:>21.0%}")

assert abs(lora_accuracy - vllm_accuracy) < 0.05, (
    f"Serving stage 3 as a runtime LoRA scores {lora_accuracy:.1%} against {vllm_accuracy:.1%} "
    "merged. These are supposed to be the same model. The usual cause is the wrong base "
    "underneath the adapter."
)

In [ ]:
# Then the cost of the LoRA path: the adapter is applied per token, so it is not free.
lora_lat, _ = run_async(run_load(EVAL_PROMPTS[:LATENCY_N], concurrency=1, ignore_eos=True,
                                 stream=True, served="stage3"))
for r in lora_lat:
    r["itl"] = (r["total"] - r["ttft"]) / (GEN_TOKENS - 1)
LORA_TTFT = [r["ttft"] for r in lora_lat]
LORA_ITL = [r["itl"] for r in lora_lat]

print(f"{'':22s}{'merged':>12s}{'runtime LoRA':>16s}{'cost':>10s}")
print("-" * 60)
for name, m, l in [("TTFT p50 (ms)", pct(VLLM_TTFT, 50) * 1000, pct(LORA_TTFT, 50) * 1000),
                   ("ITL p50 (ms)", pct(VLLM_ITL, 50) * 1000, pct(LORA_ITL, 50) * 1000)]:
    print(f"{name:22s}{m:>12.1f}{l:>16.1f}{l / m - 1:>9.0%}")

adapter_mb = sum(f.stat().st_size for f in STAGE3_DIR.rglob("*") if f.is_file()) / 1e6
print(f"\nWhat you buy with that: a second fine-tune costs {adapter_mb:.0f} MB instead of "
      f"{export_mb:.0f} MB,")
print(f"so one GPU can host {export_mb / adapter_mb:.0f}x more of them. For a platform serving many")
print("customers' adapters, that trade is not close. For a single model at high volume, merge.")

## 17. What it costs

The summary. Every number below was measured above; nothing here touches the server. Throughput
converts to money directly, and money is the argument that gets a serving change approved.

One caution about the cost line, which is the difference between a real number and a marketing one:
it divides by **peak** throughput, which assumes the GPU is saturated every second it is rented.
Nothing is. A service sized for peak traffic and idle overnight might average 15% utilization, and
its real cost per token is six times the figure below. Peak throughput sets the floor, and the
floor is not the bill.

In [ ]:
# Indicative on-demand list prices, USD/hour. Substitute yours — the ratio is the point.
GPU_HOURLY_USD = {"H100": 2.99, "A100-SXM4-80GB": 1.79, "A100": 1.10, "L40S": 0.86,
                  "V100": 0.28, "L4": 0.28, "T4": 0.11}
RATE = next((v for k, v in sorted(GPU_HOURLY_USD.items(), key=lambda kv: -len(kv[0]))
             if k.lower() in DEVICE_NAME.lower()), None)


def cost_per_million(tok_s):
    """USD per 1M output tokens at this rate, assuming the GPU is fully busy."""
    return RATE / (tok_s * 3600) * 1e6 if RATE and tok_s else float("nan")


rows = [
    ("accuracy, 200 held-out", f"{HF_ACCURACY:.1%}", f"{vllm_accuracy:.1%}"),
    ("TTFT p50 (ms)", f"{pct(HF_TTFT, 50) * 1000:.0f}", f"{pct(VLLM_TTFT, 50) * 1000:.0f}"),
    ("TTFT p95 (ms)", f"{pct(HF_TTFT, 95) * 1000:.0f}", f"{pct(VLLM_TTFT, 95) * 1000:.0f}"),
    ("ITL p50 (ms)", f"{pct(HF_ITL, 50) * 1000:.1f}", f"{pct(VLLM_ITL, 50) * 1000:.1f}"),
    ("tokens/sec, 1 request", f"{HF_DECODE_TOK_S:.0f}", f"{VLLM_DECODE_TOK_S:.0f}"),
    ("tokens/sec, peak", f"{HF_PEAK_TOK_S:.0f}", f"{VLLM_PEAK['tok_s']:.0f}"),
    ("at concurrency", f"batch {max(hf_batch, key=lambda r: r['tok_s'])['batch']}",
     str(VLLM_PEAK["concurrency"])),
    ("ragged workload, wall s", f"{HF_VARIED_WALL:.1f}", f"{VLLM_VARIED_WALL:.1f}"),
]

print(f"{DEVICE_NAME}, Llama-3.2-1B + 3 LoRAs merged, {GEN_TOKENS}-token outputs\n")
print(f"{'':26s}{'model.generate()':>18s}{'vllm serve':>14s}")
print("-" * 58)
for name, h, v in rows:
    print(f"{name:26s}{h:>18s}{v:>14s}")

if RATE:
    print(f"\ncost per 1M output tokens at ${RATE:.2f}/hr, GPU fully busy")
    print(f"  model.generate() at peak   ${cost_per_million(HF_PEAK_TOK_S):6.2f}")
    print(f"  vllm serve at peak         ${cost_per_million(VLLM_PEAK['tok_s']):6.2f}")
    print(f"  vllm serve at 15% busy     ${cost_per_million(VLLM_PEAK['tok_s']) / 0.15:6.2f}"
          "   <- the one that resembles a bill")
else:
    print(f"\nNo price for {DEVICE_NAME!r}; add one to GPU_HOURLY_USD for the cost line.")

print("\nThis is one model, one prompt shape, one GPU, one engine version, one afternoon. It is")
print("not a general claim about vLLM against transformers, and a 1B model understates the gap —")
print("bigger models spend proportionally more time memory-bound, which is what batching fixes.")

## 18. The knobs, and what each one trades

Measured above: `--max-model-len`, `--gpu-memory-utilization`, `--max-num-seqs`,
`--enable-prefix-caching`, `--enable-lora`. Not measured, but the next things to reach for, with
what each one actually costs:

| flag | buys | costs |
|---|---|---|
| `--enable-chunked-prefill` | Splits long prefills across steps so a 900-token prompt stops stalling everyone else's decode. Smoother ITL under mixed load. | Slightly worse TTFT for the request being chunked. On by default in recent versions. |
| `--kv-cache-dtype fp8` | Roughly 2x the KV cache, so roughly 2x the concurrency ceiling. | A small quality loss, and it needs sm_89+ for native fp8. |
| `--quantization awq` / `fp8` | Smaller weights means a higher decode roofline — the section 5 ceiling moves up, because there is less to read per token. | Quality loss, and a calibration step. This is the biggest single lever on the numbers above. |
| `--speculative-config` | A small draft model proposes tokens the big one verifies in a batch. Cuts ITL at low concurrency. | Wasted compute when the draft is wrong, so it *hurts* at high concurrency. |
| `--tensor-parallel-size N` | Splits the model across N GPUs when it does not fit on one. | Communication per layer. Never use it to make a model that already fits go faster. |
| `--max-num-batched-tokens` | Caps per-step work, which bounds the worst-case ITL spike. | Lower peak throughput. |

And the things no flag fixes, which is where most real serving effort actually goes:

- **Replicas and autoscaling.** Scale on **queue depth or TTFT**, not GPU utilization — a saturated
  GPU reads as 100% busy long before latency degrades, and then all at once.
- **Admission control.** Section 13's frontier flattens; past the knee, accepting more work only
  makes every request slower. Shedding load is better than serving all of it badly.
- **Request cancellation.** A client that disconnects should free its slot immediately. Without it,
  abandoned requests are paid for in full.
- **Warmup.** The first request after a deploy pays for CUDA graph capture and compilation. Send
  synthetic traffic before routing real users.
- **Model and cache versioning.** Section 11 exists because the served model can silently stop being
  the model you tested. Pin the checkpoint by digest, and re-run the accuracy gate on every deploy.

## 19. What you built

```
unsloth/Llama-3.2-1B  (base)
   ├─ notebook 01 ─► stage1-domain-lora     45 MB
   ├─ notebook 02 ─► stage2-instruct-lora   45 MB
   └─ notebook 03 ─► stage3-dpo-lora        45 MB
                          │
                          │  merge_and_unload() x3, each merge asserted
                          ▼
              serving-merged-bf16/          2.5 GB, self-contained, no PEFT
                          │
                          │  vllm serve --dtype bfloat16 --max-model-len 1280
                          ▼
              http://localhost:8000/v1/completions
                          │
                          ├─ same accuracy as notebook 03, checked before any timing
                          └─ measured: TTFT, ITL, p95, throughput against concurrency
```

The gap between notebook 03 and this one is the gap between "I fine-tuned a model" and "I shipped
one", and almost all of it is unglamorous: an artifact with the right fields in it, a process that
stays up, a number proving it still works, and a cost per token.

### The three things this notebook is built to keep you honest about

1. **A benchmark that does not fix the output length is measuring output length.** Every timed cell
   pins the token count on both sides — `min_new_tokens` on one, `ignore_eos` on the other. Without
   that, the tokens/sec comparison silently becomes a comparison of verbosity, and it will favour
   whichever system happened to answer more briefly.
2. **Faster is only a result if it is also correct.** Section 11 runs before every benchmark, and it
   reports two different things: accuracy parity, which must hold, and byte-identical agreement,
   which will not. Greedy decoding is not reproducible across engines, so a regression test that
   compares strings will fail on every deploy for reasons that do not matter.
3. **Peak throughput is not what you pay for.** The cost line in section 17 assumes the GPU is busy
   every second it is rented. The frontier in section 13 is where the actual decision lives, and a
   latency SLO is a horizontal line drawn across it.

### Honest limitations

- **Colab GPUs are shared and thermally throttled.** A single run moves several percent between
  invocations. Nothing here is repeated enough to distinguish a 5% difference from noise — this is
  the systems analogue of notebook 03's McNemar caveat, and it applies to every table above.
- **Closed-loop load, not real traffic.** The generator keeps N requests in flight and never builds
  a queue it did not intend to. Open-loop arrivals at a fixed rate are what reveal queueing
  collapse, and they are what a production load test should use.
- **One model, one prompt shape, one engine version, one GPU.** A 1B model understates vLLM's
  advantage, because a larger model spends proportionally more time memory-bound.
- **No multi-replica, no autoscaling, no gateway, no auth.** The server here listens on localhost
  with an API key it does not check. That is a demonstration, not a deployment.
- **The `p95`s are computed over 20-64 samples.** They are indicative. A tail latency number worth
  putting in an SLO needs thousands.
- **This notebook needs an L4 or A100.** Notebooks 01-03 run on the free tier; this one does not.
- **Nothing here is medical advice.** A 1B model fine-tuned on 800 abstracts is a training-mechanics
  exercise, not a clinical tool, and putting it behind an HTTP endpoint does not change that.

### Where to go next

| Change | Why |
|---|---|
| **`vllm bench serve`** | vLLM's own benchmark harness, with open-loop arrival rates and standard datasets. Use it before trusting any hand-rolled number, including these. |
| **Quantize the weights** — AWQ, GPTQ or fp8 | The single biggest lever on section 5's roofline: fewer bytes per token read means a higher decode ceiling. Start here. |
| **An open-loop load test** | Poisson arrivals at a target rate, not fixed concurrency. The only way to see queueing collapse before your users do. |
| **Speculative decoding** | A draft model cuts ITL substantially at low concurrency. Measure at high concurrency too — it can lose there. |
| **SGLang or TensorRT-LLM** | The two credible alternatives. TensorRT-LLM is usually fastest and least pleasant; SGLang wins on prefix-heavy workloads. |
| **KV cache offloading to CPU** | Trades PCIe bandwidth for concurrency when the KV cache, not compute, is the ceiling. |
| **A real gateway** | Auth, rate limits, per-tenant quotas, request logging, tracing. None of it is model work, and all of it is required. |
| **GGUF export for `llama.cpp`** | The other end of the spectrum: no GPU at all. `docs/concepts.md` §15 mentions it. |

See `docs/concepts.md` §16 for the reasoning behind every measurement in this notebook.

In [ ]:
stop_server()
print("server stopped")
print(f"\nthe artifact worth keeping: {EXPORT_DIR}")
print("copy it to Drive if you want it to survive this session:")
print(f"  !cp -r {EXPORT_DIR} /content/drive/MyDrive/finetuning-demo/")